# Electrochemical Data Analysis - Simplified

**Clean data access tool for electrochemical analysis with minimal, flexible utilities**

This notebook provides:
- **Multi-file experiment overview** via segment metadata
- **Simple utilities** for raw data access and plotting
- **Focused examples** for kinetics and current pulse analysis
- **Full Plotly freedom** for custom visualizations

---

## 🔧 Setup & Initialization

In [1]:
# Core imports
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

# Analysis system imports
from src_clean.backend.api import BackendAPI
from src_clean.analysis.registry import get_analysis_registry
from src_clean.backend.lazy_data_service import get_lazy_data_service

# Data manipulation and visualization
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 50)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
# Initialize analysis system
try:
    api = BackendAPI()
    registry = get_analysis_registry()
    lazy_service = get_lazy_data_service()
    print("✅ Analysis system initialized successfully")
    print(f"📊 Database path: {api.db.db_path}")
except Exception as e:
    print(f"❌ Failed to initialize analysis system: {e}")
    print("💡 Make sure you're running from the project directory")

✅ Analysis system initialized successfully
📊 Database path: /Users/srinathchakravarthy/PycharmProjects/Potentiostat_Data_analyser/test_data/test_database.db


## 🛠️ Utilities (Available Everywhere)

In [3]:
def get_raw_data(segment_id):
    """
    Load raw time-series data for a specific segment.
    
    Args:
        segment_id (str or int): Segment ID to load
        
    Returns:
        pd.DataFrame: Raw electrochemical data with time_s, potential_v, current_a, etc.
    """
    try:
        raw_data = lazy_service.get_segment_raw_data(str(segment_id))
        
        # Convert to pandas if it's Polars
        if hasattr(raw_data, 'to_pandas'):
            return raw_data.to_pandas()
        return raw_data
        
    except Exception as e:
        print(f"❌ Failed to load raw data for segment {segment_id}: {e}")
        return pd.DataFrame()


def plot_xy(x_col, y_col, plot_type='scatter', color_col=None, data_source=None, title=None, height=600):
    """
    Generic X vs Y plotting function.
    
    Args:
        x_col (str): X-axis column name
        y_col (str): Y-axis column name
        plot_type (str): 'scatter' or 'line'
        color_col (str): Optional color column for grouping
        data_source (pd.DataFrame): Data source (defaults to dataset_df)
        title (str): Plot title
        height (int): Plot height in pixels
    """
    # Choose data source
    if data_source is not None:
        df = data_source
    elif 'dataset_df' in globals() and dataset_df is not None:
        df = dataset_df.to_pandas() if hasattr(dataset_df, 'to_pandas') else dataset_df
    else:
        print("❌ No data source provided and no dataset_df loaded")
        return
    
    # Validate columns
    missing_cols = [col for col in [x_col, y_col] if col not in df.columns]
    if missing_cols:
        print(f"❌ Missing columns: {missing_cols}")
        print(f"Available columns: {list(df.columns)[:10]}...")
        return
    
    # Create plot
    if title is None:
        title = f"{y_col} vs {x_col}"
    
    if plot_type == 'scatter':
        fig = px.scatter(df, x=x_col, y=y_col, color=color_col, title=title, height=height)
    elif plot_type == 'line':
        fig = px.line(df, x=x_col, y=y_col, color=color_col, title=title, height=height)
    else:
        print(f"❌ Unknown plot_type: {plot_type}. Use 'scatter' or 'line'")
        return
    
    fig.update_layout(xaxis_title=x_col, yaxis_title=y_col)
    fig.show()


def plot_distribution(col, plot_type='histogram', data_source=None, title=None, height=600):
    """
    Plot distribution of a column.
    
    Args:
        col (str): Column name to plot
        plot_type (str): 'histogram', 'box', or 'violin'
        data_source (pd.DataFrame): Data source (defaults to dataset_df)
        title (str): Plot title
        height (int): Plot height in pixels
    """
    # Choose data source
    if data_source is not None:
        df = data_source
    elif 'dataset_df' in globals() and dataset_df is not None:
        df = dataset_df.to_pandas() if hasattr(dataset_df, 'to_pandas') else dataset_df
    else:
        print("❌ No data source provided and no dataset_df loaded")
        return
    
    if col not in df.columns:
        print(f"❌ Column '{col}' not found in data")
        return
    
    if title is None:
        title = f"Distribution of {col}"
    
    if plot_type == 'histogram':
        fig = px.histogram(df, x=col, title=title, height=height)
    elif plot_type == 'box':
        fig = px.box(df, y=col, title=title, height=height)
    elif plot_type == 'violin':
        fig = px.violin(df, y=col, title=title, height=height)
    else:
        print(f"❌ Unknown plot_type: {plot_type}. Use 'histogram', 'box', or 'violin'")
        return
    
    fig.show()


print("✅ Utilities defined successfully!")
print("\n🛠️  Available functions:")
print("  - get_raw_data(segment_id) → Load raw time-series data")
print("  - plot_xy(x_col, y_col, plot_type='scatter'/'line', color_col=None, data_source=None)")
print("  - plot_distribution(col, plot_type='histogram'/'box'/'violin', data_source=None)")
print("\n💡 These functions work with both segment metadata and raw data!")

✅ Utilities defined successfully!

🛠️  Available functions:
  - get_raw_data(segment_id) → Load raw time-series data
  - plot_xy(x_col, y_col, plot_type='scatter'/'line', color_col=None, data_source=None)
  - plot_distribution(col, plot_type='histogram'/'box'/'violin', data_source=None)

💡 These functions work with both segment metadata and raw data!


## 📋 Cell Selection

In [4]:
# Get all available cells
cells_data = api.get_cells()
cells_df = pd.DataFrame(cells_data)

if not cells_df.empty:
    print(f"📊 Found {len(cells_df)} cells in database:")
    print("\n" + "="*80)
    display(cells_df[['name', 'chemistry', 'capacity_ah', 'created_at']].head(10))
    print("="*80)
    
    # Show cell names for easy copying
    print("\n🎯 Available cell names for selection:")
    for i, cell_name in enumerate(cells_df['name'].head(10), 1):
        print(f"  {i}. {cell_name}")
else:
    print("❌ No cells found in database")
    print("💡 Process some files first using the UI")

📊 Found 3 cells in database:



,name,chemistry,capacity_ah,created_at
0,test_versastudio,Li_ion,0.15600,2025-09-12 15:09:05
1,AR3677,Li_metal,0.14518,2025-09-10 21:44:08
2,AR3753,Li_metal,0.14552,2025-09-10 21:42:04



🎯 Available cell names for selection:
  1. test_versastudio
  2. AR3677
  3. AR3753


In [5]:
# 🎯 USER SELECTION: Change this cell name to analyze different data
SELECTED_CELL = "AR3753"  # 👈 CHANGE THIS TO YOUR CELL NAME

print(f"🔍 Selected cell: {SELECTED_CELL}")

# Verify cell exists
if SELECTED_CELL in cells_df['name'].values:
    selected_cell_info = cells_df[cells_df['name'] == SELECTED_CELL].iloc[0]
    print("\n📋 Cell Information:")
    print(f"  Name: {selected_cell_info['name']}")
    print(f"  Chemistry: {selected_cell_info.get('chemistry', 'N/A')}")
    print(f"  Capacity: {selected_cell_info.get('capacity_ah', 'N/A')} Ah")
    print(f"  Created: {selected_cell_info['created_at']}")
    
    # Get files for this cell
    cell_files = api.get_cell_files(SELECTED_CELL)
    print(f"\n📁 Files in cell: {len(cell_files)}")
    for file_info in cell_files:
        print(f"  - {file_info['original_filename']} (ID: {file_info['file_id']})")
else:
    print(f"❌ Cell '{SELECTED_CELL}' not found in database")
    print("💡 Check the cell name spelling or select from the list above")

🔍 Selected cell: AR3753

📋 Cell Information:
  Name: AR3753
  Chemistry: Li_metal
  Capacity: 0.14552 Ah
  Created: 2025-09-10 21:42:04

📁 Files in cell: 1
  - AR-3753_3_electrode_full_GITT_EIS_0_04_MB_C06.mpr (ID: AR3753_AR-3753_3_electrode_full_GITT_EIS_0_04_MB_C06_20250918_092107)


## 📊 Load Segment Metadata & Column Categorization

In [6]:
# Load comprehensive dataset (segment metadata + analytics)
print(f"🔄 Loading experiment data for cell: {SELECTED_CELL}")
print("   This provides multi-file experiment overview via segment metadata...")

try:
    # Load segment metadata with embedded analytics
    dataset = api.get_research_dataset_for_perspective(cells=[SELECTED_CELL])
    
    # Convert to pandas for easy manipulation
    if hasattr(dataset, 'to_pandas'):
        dataset_df = dataset.to_pandas()
    else:
        dataset_df = dataset
    
    print(f"✅ Experiment data loaded successfully!")
    print(f"📊 Shape: {dataset_df.shape[0]} segments × {dataset_df.shape[1]} columns")
    
    # Quick overview
    time_span = dataset_df['end_time_s'].max() - dataset_df['start_time_s'].min()
    print(f"⏱️  Total experiment time: {time_span:.1f} seconds ({time_span/3600:.1f} hours)")
    print(f"⚡ Voltage range: {dataset_df['start_potential_v'].min():.3f}V to {dataset_df['end_potential_v'].max():.3f}V")
    print(f"🔋 Current range: {dataset_df['start_current_a'].min():.6f}A to {dataset_df['end_current_a'].max():.6f}A")
    
    # Show techniques present
    techniques = dataset_df['fundamental_technique'].value_counts()
    print(f"\n🧪 Techniques present:")
    for technique, count in techniques.items():
        print(f"  - {technique}: {count} segments")
    
    # Show files present
    if 'original_filename' in dataset_df.columns:
        files = dataset_df['original_filename'].value_counts()
        print(f"\n📁 Files in experiment:")
        for filename, count in files.items():
            print(f"  - {filename}: {count} segments")
        
except Exception as e:
    print(f"❌ Failed to load experiment data: {e}")
    dataset_df = None

🔄 Loading experiment data for cell: AR3753
   This provides multi-file experiment overview via segment metadata...
API call 119 segments
First segment keys: ['id', 'file_id', 'segment_index', 'technique_id', 'technique_name', 'fundamental_technique', 'start_row', 'end_row', 'start_time_s', 'end_time_s', 'point_count', 'duration_s', 'start_potential_v', 'end_potential_v', 'start_current_a', 'end_current_a', 'capacity_ah', 'energy_wh', 'start_timestamp', 'capacity_cumulative_ah', 'energy_cumulative_wh', 'charge_cumulative_ah', 'discharge_cumulative_ah', 'energy_charge_cumulative_wh', 'energy_discharge_cumulative_wh', 'capacity_absolute_cumulative_ah', 'energy_absolute_cumulative_wh', 'analysis_status', 'analysis_results', 'segment_metadata', 'cell_id', 'cell_name', 'created_at', 'exp_discharge_energy_wh', 'exp_discharge_cap_ah', 'exp_charge_cap_ah', 'exp_time_cumulative_s', 'exp_charge_energy_wh', 'start_we_potential_v', 'end_we_potential_v', 'start_ce_potential_v', 'end_ce_potential_v',

/Users/srinathchakravarthy/PycharmProjects/Potentiostat_Data_analyser/src_clean/backend/api.py:2790: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_df[col] = final_df[col].fillna(False).astype('bool')


✅ Experiment data loaded successfully!
📊 Shape: 119 segments × 195 columns
⏱️  Total experiment time: 571207.4 seconds (158.7 hours)
⚡ Voltage range: 0.542V to 4.250V
🔋 Current range: -0.048023A to 0.048391A

🧪 Techniques present:
  - rest: 60 segments
  - galvanostatic: 31 segments
  - eis: 28 segments

📁 Files in experiment:
  - AR-3753_3_electrode_full_GITT_EIS_0_04_MB_C06.mpr: 119 segments


In [7]:
# Categorize columns for easy discovery
if dataset_df is not None:
    all_columns = list(dataset_df.columns)
    
    # Column categories
    time_cols = [col for col in all_columns if 'time' in col.lower()]
    voltage_cols = [col for col in all_columns if any(x in col.lower() for x in ['potential', 'voltage'])]
    current_cols = [col for col in all_columns if 'current' in col.lower()]
    capacity_cols = [col for col in all_columns if 'capacity' in col.lower()]
    energy_cols = [col for col in all_columns if 'energy' in col.lower()]
    analytics_cols = [col for col in all_columns if any(x in col for x in ['time_constant', 'analytics', 'r_squared'])]
    technique_cols = [col for col in all_columns if 'technique' in col.lower()]
    
    print("🗂️  Column Categories for Plotting:")
    print(f"\n⏱️  Time columns ({len(time_cols)}):")
    for col in time_cols[:8]:  
        print(f"    {col}")
    
    print(f"\n⚡ Voltage columns ({len(voltage_cols)}):")
    for col in voltage_cols[:8]:
        print(f"    {col}")
    
    print(f"\n🔋 Current columns ({len(current_cols)}):")
    for col in current_cols[:8]:
        print(f"    {col}")
    
    print(f"\n🔋 Capacity columns ({len(capacity_cols)}):")
    for col in capacity_cols[:8]:
        print(f"    {col}")
    
    print(f"\n📊 Analytics columns ({len(analytics_cols)}):")
    for col in analytics_cols[:10]:
        print(f"    {col}")
    
    print(f"\n🧪 Technique columns ({len(technique_cols)}):")
    for col in technique_cols:
        print(f"    {col}")
    
    print(f"\n💡 Use these column names in plot_xy() and plot_distribution() functions!")

🗂️  Column Categories for Plotting:

⏱️  Time columns (17):
    start_time_s
    end_time_s
    start_timestamp
    exp_time_cumulative_s
    basic_statistics_analytics_start_time_s
    basic_statistics_analytics_end_time_s
    basic_statistics_analytics_start_timestamp
    basic_statistics_analytics_exp_time_cumulative_s

⚡ Voltage columns (38):
    start_potential_v
    end_potential_v
    start_we_potential_v
    end_we_potential_v
    start_ce_potential_v
    end_ce_potential_v
    basic_statistics_analytics_start_potential_v
    basic_statistics_analytics_end_potential_v

🔋 Current columns (5):
    start_current_a
    end_current_a
    basic_statistics_analytics_start_current_a
    basic_statistics_analytics_end_current_a
    resistance_analytics_average_current_a

🔋 Capacity columns (6):
    capacity_ah
    capacity_cumulative_ah
    capacity_absolute_cumulative_ah
    basic_statistics_analytics_capacity_ah
    basic_statistics_analytics_capacity_cumulative_ah
    basic_statistic

## 🎨 Analysis Examples

### Example 1: Kinetics Analysis

In [12]:
kinetics_cols = [col for col in dataset_df.columns if 'ki' in col]
kinetics_cols

['kinetics_analytics_voltage_infinity',
 'kinetics_analytics_voltage_amplitude',
 'kinetics_analytics_time_constant_s',
 'kinetics_analytics_we_voltage_infinity',
 'kinetics_analytics_we_voltage_amplitude',
 'kinetics_analytics_we_time_constant_s',
 'kinetics_analytics_ce_voltage_infinity',
 'kinetics_analytics_ce_voltage_amplitude',
 'kinetics_analytics_ce_time_constant_s',
 'kinetics_analytics_r_squared',
 'kinetics_analytics_rmse',
 'kinetics_analytics_fit_type',
 'kinetics_analytics_selection_reason',
 'kinetics_analytics_extraction_quality_exp',
 'kinetics_analytics_extraction_quality_sqrt',
 'kinetics_analytics_fit_quality',
 'kinetics_analytics_diffusion_regime',
 'kinetics_analytics_analysis_type',
 'kinetics_analytics_quality_score',
 'kinetics_analytics_summary_total_segments',
 'kinetics_analytics_summary_high_quality_fits',
 'kinetics_analytics_summary_fit_success_rate',
 'kinetics_analytics_insight_dominant_process',
 'kinetics_analytics_insight_data_quality',
 'kinetics_a

In [8]:
# Kinetics Analysis - Time Constants vs Time
print("📊 Kinetics Analysis - Time Constants")

if dataset_df is not None:
    # Find kinetics analytics columns
    kinetics_cols = [col for col in dataset_df.columns if 'kinetics_analytics' in col and 'time_constant' in col]
    exp_cols = [col for col in kinetics_cols if 'exponential' in col]
    sqrt_cols = [col for col in kinetics_cols if 'sqrt' in col]
    
    print(f"Found kinetics columns:")
    print(f"  Exponential: {exp_cols}")
    print(f"  Sqrt(t): {sqrt_cols}")
    
    if exp_cols:
        exp_col = exp_cols[0]  # Take first exponential time constant column
        
        # Filter data where kinetics analytics are available
        kinetics_data = dataset_df[dataset_df[exp_col].notna()]
        
        if not kinetics_data.empty:
            print(f"\n📈 Plotting {len(kinetics_data)} segments with kinetics data")
            
            # Plot exponential time constants vs time
            plot_xy('start_time_s', exp_col, 
                   plot_type='scatter', 
                   color_col='fundamental_technique',
                   data_source=kinetics_data,
                   title="Kinetics: Exponential Time Constants vs Time")
        else:
            print("❌ No kinetics data available in this dataset")
    
    if sqrt_cols:
        sqrt_col = sqrt_cols[0]  # Take first sqrt time constant column
        
        # Filter data where sqrt kinetics analytics are available
        sqrt_data = dataset_df[dataset_df[sqrt_col].notna()]
        
        if not sqrt_data.empty:
            print(f"\n📈 Plotting sqrt(t) time constants for {len(sqrt_data)} segments")
            
            # Plot sqrt(t) time constants vs time
            plot_xy('start_time_s', sqrt_col, 
                   plot_type='scatter', 
                   color_col='fundamental_technique',
                   data_source=sqrt_data,
                   title="Kinetics: Sqrt(t) Time Constants vs Time")
    
    if not kinetics_cols:
        print("ℹ️  No kinetics analytics columns found in dataset")
        print("💡 Available analytics columns:", [col for col in dataset_df.columns if 'analytics' in col][:5])

📊 Kinetics Analysis - Time Constants
Found kinetics columns:
  Exponential: []
  Sqrt(t): []


### Example 2: Current Pulse Analysis

In [ ]:
# Current Pulse Analysis - Time Constants vs Time
print("📊 Current Pulse Analysis - Time Constants")

if dataset_df is not None:
    # Find current pulse analytics columns
    pulse_cols = [col for col in dataset_df.columns if 'current_pulse' in col and 'time_constant' in col]
    pulse_exp_cols = [col for col in pulse_cols if 'exponential' in col]
    pulse_sqrt_cols = [col for col in pulse_cols if 'sqrt' in col]
    
    print(f"Found current pulse columns:")
    print(f"  Exponential: {pulse_exp_cols}")
    print(f"  Sqrt(t): {pulse_sqrt_cols}")
    
    if pulse_exp_cols:
        pulse_exp_col = pulse_exp_cols[0]  # Take first exponential time constant column
        
        # Filter data where current pulse analytics are available
        pulse_data = dataset_df[dataset_df[pulse_exp_col].notna()]
        
        if not pulse_data.empty:
            print(f"\n📈 Plotting {len(pulse_data)} segments with current pulse data")
            
            # Plot exponential time constants vs time
            plot_xy('start_time_s', pulse_exp_col, 
                   plot_type='scatter', 
                   color_col='fundamental_technique',
                   data_source=pulse_data,
                   title="Current Pulse: Exponential Time Constants vs Time")
        else:
            print("❌ No current pulse data available in this dataset")
    
    if pulse_sqrt_cols:
        pulse_sqrt_col = pulse_sqrt_cols[0]  # Take first sqrt time constant column
        
        # Filter data where sqrt current pulse analytics are available
        pulse_sqrt_data = dataset_df[dataset_df[pulse_sqrt_col].notna()]
        
        if not pulse_sqrt_data.empty:
            print(f"\n📈 Plotting sqrt(t) time constants for {len(pulse_sqrt_data)} segments")
            
            # Plot sqrt(t) time constants vs time
            plot_xy('start_time_s', pulse_sqrt_col, 
                   plot_type='scatter', 
                   color_col='fundamental_technique',
                   data_source=pulse_sqrt_data,
                   title="Current Pulse: Sqrt(t) Time Constants vs Time")
    
    if not pulse_cols:
        print("ℹ️  No current pulse analytics columns found in dataset")
        print("💡 Available analytics columns:", [col for col in dataset_df.columns if 'analytics' in col][:5])

### Example 3: Raw Data Access

In [ ]:
# Example: Load and plot raw data for a specific segment
print("📊 Raw Data Access Example")

if dataset_df is not None and len(dataset_df) > 0:
    # Get first segment as example
    example_segment = dataset_df.iloc[0]
    segment_id = example_segment['id']
    
    print(f"🔍 Loading raw data for segment {segment_id}")
    print(f"   Technique: {example_segment['fundamental_technique']}")
    print(f"   Duration: {example_segment['duration_s']:.1f}s")
    print(f"   Points: {example_segment['point_count']}")
    
    # Load raw data using utility function
    raw_data = get_raw_data(segment_id)
    
    if not raw_data.empty:
        print(f"\n✅ Raw data loaded: {raw_data.shape[0]} points × {raw_data.shape[1]} columns")
        
        # Show available columns
        time_cols = [col for col in raw_data.columns if 'time' in col.lower()]
        voltage_cols = [col for col in raw_data.columns if any(x in col.lower() for x in ['potential', 'voltage'])]
        current_cols = [col for col in raw_data.columns if 'current' in col.lower()]
        
        print(f"\n📋 Available for plotting:")
        print(f"   Time: {time_cols[:3]}")
        print(f"   Voltage: {voltage_cols[:3]}")
        print(f"   Current: {current_cols[:3]}")
        
        # Plot example if suitable columns exist
        if time_cols and voltage_cols:
            print(f"\n📈 Plotting {voltage_cols[0]} vs {time_cols[0]}")
            plot_xy(time_cols[0], voltage_cols[0], 
                   plot_type='line', 
                   data_source=raw_data,
                   title=f"Raw Data - Segment {segment_id}")
            
            print(f"\n💡 To plot other segments: raw_data = get_raw_data('SEGMENT_ID')")
        else:
            print("❌ No suitable time/voltage columns found for plotting")
    else:
        print("❌ Failed to load raw data")
else:
    print("❌ No segment data available")

## 🎯 User Playground

**Your space for custom analysis and plotting!**

Use the utility functions or write your own Plotly code directly.

In [ ]:
# 🎨 YOUR CUSTOM ANALYSIS SPACE

# Quick data overview
if 'dataset_df' in globals() and dataset_df is not None:
    print(f"📊 Dataset available: {dataset_df.shape[0]} segments × {dataset_df.shape[1]} columns")
    print(f"🧪 Techniques: {list(dataset_df['fundamental_technique'].unique())}")
    
    # Show some useful columns for plotting
    plot_cols = [col for col in dataset_df.columns if any(x in col.lower() for x in 
                ['time', 'potential', 'current', 'capacity', 'analytics'])]
    print(f"\n📈 Useful columns for plotting ({len(plot_cols)}):")
    for i, col in enumerate(plot_cols[:15], 1):
        print(f"  {i:2d}. {col}")
    
    print(f"\n💡 Example usage:")
    print(f"   plot_xy('start_time_s', 'start_potential_v', color_col='fundamental_technique')")
    print(f"   plot_distribution('duration_s')")
    print(f"   raw_data = get_raw_data('{dataset_df.iloc[0]['id']}')")
else:
    print("❌ No dataset loaded. Run the data loading section first.")

In [ ]:
# 🎯 EXAMPLE: Experiment overview plots

# Uncomment and modify these examples:

# 1. Voltage progression over time
# plot_xy('start_time_s', 'start_potential_v', color_col='fundamental_technique', 
#        title="Experiment Overview: Voltage vs Time")

# 2. Capacity progression
# plot_xy('start_time_s', 'capacity_cumulative_ah', color_col='original_filename',
#        title="Capacity Progression Across Files")

# 3. Duration distribution by technique
# plot_distribution('duration_s', plot_type='box')

print("💡 Uncomment and modify the examples above, or write your own!")

In [ ]:
# 🎯 ADVANCED: Direct Plotly for full control

# You can also use Plotly directly for complete control:

# import plotly.express as px
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# Example: Multi-subplot analysis
# fig = make_subplots(rows=2, cols=2, 
#                    subplot_titles=['Voltage vs Time', 'Current vs Time', 
#                                   'Capacity vs Time', 'Duration Distribution'])
# 
# # Add your traces here...
# fig.show()

print("💡 Use direct Plotly for advanced multi-plot analysis!")

---

## ✅ Analysis Complete!

**This simplified notebook provides:**

### 🎯 Key Features:
- ✅ **Multi-file experiment overview** via segment metadata
- ✅ **Simple utilities** available everywhere
- ✅ **Focused analytics examples** (kinetics, current pulse)
- ✅ **Raw data access** for detailed analysis
- ✅ **Full Plotly freedom** for custom visualizations

### 🚀 Usage Patterns:
1. **Overview Analysis**: Use `dataset_df` for experiment trends
2. **Detailed Analysis**: Use `get_raw_data(segment_id)` for specific segments
3. **Quick Plots**: Use `plot_xy()` and `plot_distribution()` utilities
4. **Advanced Plots**: Write custom Plotly code in User Playground

### 💡 Next Steps:
- Modify `SELECTED_CELL` to analyze different experiments
- Use column categorization to discover plotting options
- Combine segment overview with raw data drill-down
- Extend with your own analysis functions

---

**Happy analyzing! 🚀**